# VCuboidFIT — benchmark pitching, pitch-v2
Runtime GPU **A100**. Mini = minh hoạ, in-sample; pool = kết quả chính, T mở một lần.
Không train, không VLM. Mặc định **RUN_T = False**. Chạy từng ô theo thứ tự.
Upload notebook bằng File → Upload notebook; code lấy từ ZIP HEAD local, chưa push.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, hashlib, zipfile, subprocess, tempfile, shutil, time, threading
from datetime import datetime, timezone

drive.mount('/content/drive')
REPO_SHA = 'f4a2abf4f33c09011876deeeb0fe0e523d9b48b8'  # thay bằng repo_sha trong manifest
DATA_ZIP = Path('/content/drive/MyDrive/vcf_pack/pool_v24/manifest.json')
MINI_DATA_ZIP = Path('/content/drive/MyDrive/datatest/vcf_manifest.json')
CODE_ZIP = Path('/content/drive/MyDrive/vcf_pack') / f'code_{REPO_SHA[:7]}.zip'
RESULTS = Path('/content/drive/MyDrive/vcf_results')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-' + os.urandom(4).hex()
SAVE = RESULTS / RUN_ID
SAVE.mkdir(parents=True, exist_ok=False)
RUN = Path(tempfile.mkdtemp(prefix='vcf-pitch-', dir='/content'))
REPO = RUN / 'repo'
os.environ['PYTHONIOENCODING'] = 'utf-8'
EXPECTED_PREREG = '590a7a6a17fd55b63f778dbd79c4b01ed6f33add705d84225d7db7b890a0a75d'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
print(gpu)
if 'A100' not in gpu:
    raise RuntimeError('Chọn runtime A100 trước khi benchmark K1')
print('RUN_ID', RUN_ID, 'REPO_SHA', REPO_SHA)


## Đo thời gian và VRAM từng bước; lưu ngay lên Drive, cả khi lệnh lỗi


In [ ]:
TIMING = []
def measured(name, action):
    stop = threading.Event()
    samples = []
    def monitor():
        while not stop.is_set():
            try:
                used = subprocess.check_output(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], text=True)
                samples.append(max(float(v) for v in used.splitlines()))
            except (OSError, subprocess.SubprocessError, ValueError):
                pass
            stop.wait(1.0)
    thread = threading.Thread(target=monitor, daemon=True)
    thread.start()
    started = time.perf_counter()
    status = 0
    try:
        return action()
    except BaseException as exc:
        status = getattr(exc, 'returncode', 1)
        raise
    finally:
        stop.set()
        thread.join()
        TIMING.append(dict(step=name, elapsed_s=time.perf_counter()-started,
                           exit_code=status, gpu_peak_used_mib=max(samples) if samples else None,
                           gpu_samples=len(samples), gpu_measurement='nvidia-smi device-wide, 1s sampling'))
        (SAVE/'timing.json').write_text(json.dumps(dict(run_id=RUN_ID, repo_sha=REPO_SHA, gpu=gpu,
                                                       steps=TIMING), indent=2), encoding='utf-8')
        print(TIMING[-1])
def command(name, args):
    def execute():
        with (SAVE/(name+'.log')).open('w', encoding='utf-8') as log:
            log.write(' '.join(map(str,args))+'\n')
            log.flush()
            completed = subprocess.run(list(map(str,args)), cwd=REPO/'model/worker',
                                       stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
        print((SAVE/(name+'.log')).read_text(encoding='utf-8')[-4000:])
        completed.check_returncode()
    return measured(name, execute)
def digest(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()
def extract_checked(zpath, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for item in z.infolist():
            dest = (target/item.filename).resolve()
            if not dest.is_relative_to(target.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Unsafe ZIP entry')
            if dest.exists() and not item.is_dir():
                raise ValueError('Duplicate ZIP entry: '+item.filename)
        z.extractall(target)


## Code từ HEAD local, kiểm SHA và cài worker[gpu]


In [ ]:
pool_manifest = json.loads(DATA_ZIP.read_text(encoding='utf-8-sig'))
assert pool_manifest['repo_sha'] == REPO_SHA
assert CODE_ZIP.name == pool_manifest['code']['name']
assert digest(CODE_ZIP) == pool_manifest['code']['sha256']
with zipfile.ZipFile(CODE_ZIP) as z:
    assert z.comment.decode().strip() == REPO_SHA, 'Git archive SHA mismatch'
measured('code_extract', lambda: extract_checked(CODE_ZIP, REPO))
os.environ['PYTHONPATH'] = str(REPO/'model/worker')
command('install_gpu', [sys.executable, '-m', 'pip', 'install', '-e', str(REPO/'model/worker')+'[gpu]'])
command('bench_cli_contract', [sys.executable, '-m', 'c4.cli.bench_pitch', '--help'])
print('CODE SHA OK', REPO_SHA)


## Ghép part theo manifest, kiểm SHA trước khi giải nén
Các part là ZIP độc lập có lớp `data/`; ghép cây thư mục, không nối byte ZIP.
Mini dùng manifest vcf-pack đủ 6 camera; không lấy mini từ pool.


In [ ]:
def unpack_manifest(path, target, pool=False):
    manifest = json.loads(path.read_text(encoding='utf-8-sig'))
    parts = manifest['parts']
    file_count = 0
    for part in parts:
        name = part.get('name', part.get('file', part.get('filename')))
        if not name or Path(name).name != name:
            raise ValueError('Invalid part name')
        archive = path.parent/name
        assert digest(archive) == part['sha256'], name+' SHA mismatch'
        size = part.get('size_bytes', part.get('bytes', part.get('size')))
        if size is not None:
            assert archive.stat().st_size == size
        if pool:
            assert archive.stat().st_size <= 2000000000
        with zipfile.ZipFile(archive) as z:
            count = sum(not f.is_dir() for f in z.infolist())
            if 'file_count' in part:
                assert count == part['file_count']
            file_count += count
        extract_checked(archive, target)
    if 'file_count' in manifest:
        assert file_count == manifest['file_count']
    roots = [p.parent for p in target.rglob('scene.json') if p.parent.name.startswith('v1.0-')]
    assert len(roots) == 1, 'Expected one nuScenes metadata version'
    data = roots[0].parent
    assert (data/'samples/LIDAR_TOP').is_dir()
    assert len(list((data/'samples').glob('CAM_*'))) == 6, 'Need all six cameras'
    if pool:
        assert roots[0].name == 'v1.0-trainval'
        assert manifest['model_mode'] == 'cluster_only'
        assert manifest['keyframes'] == 6753 and manifest['camera_files'] == 40518
        assert not (data/'sweeps').exists()
    else:
        assert roots[0].name == 'v1.0-mini'
    print('DATA OK', data, file_count)
    return data
MINI_ROOT = measured('mini_unpack', lambda: unpack_manifest(MINI_DATA_ZIP, RUN/'mini_data'))
POOL_ROOT = measured('pool_unpack', lambda: unpack_manifest(DATA_ZIP, RUN/'pool_data', pool=True))


## Cổng prereg pitch-v2 — sai hash thì dừng trước T


In [ ]:
prereg = pool_manifest['prereg']
assert hashlib.sha256(prereg.encode('utf-8')).hexdigest() == EXPECTED_PREREG
assert pool_manifest['prereg_sha256'] == EXPECTED_PREREG
(SAVE/'prereg.txt').write_text(prereg, encoding='utf-8')
(SAVE/'prereg_sha256.txt').write_text(EXPECTED_PREREG+'\n', encoding='utf-8')
print('PREREG OK', EXPECTED_PREREG)


## Chuỗi §6 — mini minh hoạ in-sample; pool dừng trước benchmark
Cluster-only, không sweeps hay PointPillars train/infer. Context mini fit trên toàn mini (nhãn V tạm trong manifest context).
Rare mini dùng ref=all/pool=all. Pool dùng V/T từ lidar_index, không đổi chia hay trọng số.
GT RHCIO/eval/biểu đồ do CLI bench_pitch quản lý, không import nội bộ.


In [ ]:
import pandas as pd
MINI_JOB = RUN/'mini_job'
POOL_JOB = RUN/'pool_job'
def prepare(mode, data, job):
    common = ['--job-dir', str(job), '--data-root', str(data)]
    def cli(step, module, extra=()):
        command(mode+'_'+step, [sys.executable, '-m', 'c4.cli.'+module, *extra, *common])
    cli('index', 'lidar_index')
    lid = pd.read_parquet(job/'lidar/index.parquet')
    if mode == 'mini':
        assert len(lid) == 404 and lid['scene_name'].nunique() == 10
    else:
        assert len(lid) == 6753 and lid['scene_name'].nunique() == 170
        assert lid.groupby('split').size().to_dict() == dict(T=1389,V=3973,S=279,P=1112)
    split = lid.loc[:, [lid.columns[0], 'split']].copy()
    if mode == 'mini':
        split['split'] = 'V'  # context illustration fit on all mini, in-sample
    split.to_csv(job/'split_manifest.csv', index=False)
    cli('tier0', 'lidar_t0')
    cli('clip', 'extract_clip', ['--device', 'cuda'])
    cli('context_cv', 'context', ['cv'])
    cli('context_fit', 'context', ['fit', '--split', 'V'])
    cli('context_run', 'context', ['run'])
    splits = ['--pool-split', 'all', '--ref-split', 'all'] if mode == 'mini' else ['--pool-split', 'T', '--ref-split', 'V']
    cli('rare_lidar', 'rare', ['run', '--profile', 'lidar_only', *splits])
    cli('rare_crops', 'rare', ['crops', '--device', 'cuda'])
    cli('rare_multisensor', 'rare', ['run', '--profile', 'multisensor', *splits])
def save_job(mode, job):
    measured(mode+'_save', lambda: shutil.copytree(job, SAVE/mode/'job', dirs_exist_ok=True))
prepare('mini', MINI_ROOT, MINI_JOB)
MINI_OUT = SAVE/'mini'
MINI_OUT.mkdir(exist_ok=True)
try:
    command('mini_bench', [sys.executable, '-m', 'c4.cli.bench_pitch', '--job-dir', MINI_JOB,
                           '--mode', 'mini', '--out', MINI_OUT])
finally:
    save_job('mini', MINI_JOB)


In [ ]:
def check_t_closed():
    assert hashlib.sha256(prereg.encode('utf-8')).hexdigest() == EXPECTED_PREREG
    if (RESULTS/'pitch-v2-T-opened.json').exists():
        raise RuntimeError('T_ALREADY_OPENED: Drive marker exists; stop')
    for report in RESULTS.rglob('report_pitch.json'):
        content = json.loads(report.read_text(encoding='utf-8'))
        if content.get('mode') == 'pool':
            raise RuntimeError('T_ALREADY_OPENED: '+str(report))
check_t_closed()
prepare('pool', POOL_ROOT, POOL_JOB)
save_job('pool', POOL_JOB)
print('POOL PREPARED. STOP before bench_pitch --mode pool; review hashes and mini results.')


## T — người dùng bật tay một lần sau khi xem mini và cấu hình đóng băng
Run all giữ T đóng. Marker Drive ghi trước lệnh, giữ cả khi lỗi/gián đoạn.
Exit 2/3: dừng và chuyển log cho người duyệt; không xoá marker hay sửa prereg để thử lại.


In [ ]:
RUN_T = False  # @param {type:"boolean"}
if RUN_T:
    check_t_closed()
    POOL_OUT = SAVE/'pool'
    POOL_OUT.mkdir(exist_ok=True)
    marker = RESULTS/'pitch-v2-T-opened.json'
    with marker.open('x', encoding='utf-8') as f:
        json.dump(dict(run_id=RUN_ID, repo_sha=REPO_SHA, prereg_sha256=EXPECTED_PREREG,
                       opened_utc=datetime.now(timezone.utc).isoformat()), f, indent=2)
    try:
        command('pool_bench', [sys.executable, '-m', 'c4.cli.bench_pitch', '--job-dir', POOL_JOB,
                               '--mode', 'pool', '--out', POOL_OUT])
    finally:
        save_job('pool', POOL_JOB)
else:
    print('RUN_T=False: T chưa mở; chuẩn bị và timing đã lưu tại', SAVE)
